## ETL Pipeline: Silver to Gold Layer

Este pipeline é responsável pela materialização da camada **Gold**.
A arquitetura segue o princípio de **Imutabilidade da Bronze** e **Idempotência da Silver**.

**Decisões de Arquitetura:**
1.  **Origem:** Banco de dados postgres da camada SIlver.
2.  **Destino:** Schema DW no baco postgres.
3.  **Modelagem:** Criação de uma *Fato* e **4 Dimensções**.

In [37]:
import pandas as pd
import numpy as np
import hashlib
import psycopg2
import os
from psycopg2.extras import execute_values
import warnings

pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

DDL_FILE = os.path.join('..', 'Data Layer', 'gold', 'ddl.sql')

DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'brazilian-e-commerce',
    'user': 'admin',
    'password': 'admin123'
}

In [38]:
print("Conectando no Banco para ler a Silver")


conn = psycopg2.connect(**DB_CONFIG)

query_silver = """ 
    SELECT
        sk_order_item, 
        order_id, 
        order_item_id, 
        product_id, 
        seller_id,
        price, 
        freight_value, 
        total_item_value, 
        product_category_name, 
        price_segment,
        order_status, 
        order_purchase_timestamp, 
        days_to_deliver 
        FROM silver.sales_order_items
"""

df_silver = pd.read_sql(query_silver, conn)

df_silver['order_purchase_timestamp'] = pd.to_datetime(df_silver['order_purchase_timestamp'])

print(f"Leitura concluída: {len(df_silver):,} registros recuperados da Silver.")

Conectando no Banco para ler a Silver
Leitura concluída: 112,650 registros recuperados da Silver.


In [39]:
def gerar_srk(valor):
    """Gera Surrogate Key"""
    
    if pd.isna(valor) or valor == '':
        
        return None
    
    return hashlib.md5(str(valor).encode('utf-8')).hexdigest()

def executar_script_sql(arquivo_sql, cursor):
    """Lê e executa arquivo SQL externo"""
    
    try:
        with open(arquivo_sql, 'r', encoding='utf-8') as f:
            
            sql_script = f.read()
            cursor.execute(sql_script)
            
            print(f"Script DDL executado: {os.path.basename(arquivo_sql)}")
            
    except Exception as e:
        print(f"Erro ao executar SQL: {e}")
        raise e

In [40]:
print("Transformando Dimensões")

df_dim_prd = df_silver[['product_id', 
               'product_category_name', 
               'price']].drop_duplicates(subset=['product_id'])

df_dim_prd['srk_prd'] = df_dim_prd['product_id'].apply(gerar_srk)

df_dim_prd = df_dim_prd.rename(columns={
    'product_id': 'cod_prd', 
    'product_category_name': 'nam_cat', 
    'price': 'nam_sgm_prc'
})

df_dim_prd = df_dim_prd[['srk_prd', 
                         'cod_prd', 
                         'nam_cat', 
                         'nam_sgm_prc']]

Transformando Dimensões


In [41]:
print("Criando DIM_VDR")

df_dim_vdr = df_silver[['seller_id']].drop_duplicates(subset=['seller_id'])

df_dim_vdr['srk_vdr'] = df_dim_vdr['seller_id'].apply(gerar_srk)

df_dim_vdr = df_dim_vdr.rename(columns={'seller_id': 'cod_vdr'})

df_dim_vdr = df_dim_vdr[['srk_vdr', 
                         'cod_vdr']]

Criando DIM_VDR


In [42]:
print("Criando DIM_STS")

df_dim_sts = df_silver[['order_status']].drop_duplicates(subset=['order_status'])

group_map = {
    'delivered': 'Concluido', 
    'shipped': 'Transito',
    'created': 'Novo', 
    'approved': 'Novo',
    'invoiced': 'Processando', 
    'processing': 'Processando',
    'canceled': 'Insucesso', 
    'unavailable': 'Insucesso'
}

finished_statuses = ['delivered', 
                     'canceled', 
                     'unavailable']

df_dim_sts['nam_grp_sts'] = df_dim_sts['order_status'].map(group_map)

df_dim_sts['flg_fin'] = df_dim_sts['order_status'].isin(finished_statuses)

df_dim_sts['srk_sts'] = df_dim_sts['order_status'].apply(gerar_srk)

df_dim_sts = df_dim_sts.rename(columns={'order_status': 'nam_sts'})

df_dim_sts = df_dim_sts[['srk_sts', 
                         'nam_sts', 
                         'nam_grp_sts', 
                         'flg_fin']]


Criando DIM_STS


In [43]:
print("Criando DIM_TMP")

datas_unicas = df_silver['order_purchase_timestamp'].dt.normalize().unique()

df_dim_tmp = pd.DataFrame({'dat_ref': datas_unicas})

df_dim_tmp['srk_tmp'] = df_dim_tmp['dat_ref'].dt.strftime('%Y%m%d').astype(int)
df_dim_tmp['num_ano'] = df_dim_tmp['dat_ref'].dt.year
df_dim_tmp['num_mes'] = df_dim_tmp['dat_ref'].dt.month
df_dim_tmp['nam_mes'] = df_dim_tmp['dat_ref'].dt.month_name()
df_dim_tmp['num_tri'] = df_dim_tmp['dat_ref'].dt.quarter
df_dim_tmp['nam_sem'] = df_dim_tmp['dat_ref'].dt.isocalendar().week.astype(int)
df_dim_tmp['nam_dia_sem'] = df_dim_tmp['dat_ref'].dt.day_name()
df_dim_tmp['flg_fim_sem'] = df_dim_tmp['dat_ref'].dt.dayofweek >= 5

df_dim_tmp = df_dim_tmp[['srk_tmp', 
                         'dat_ref', 
                         'num_ano', 
                         'num_mes', 
                         'nam_mes', 
                         'num_tri', 
                         'nam_sem', 
                         'nam_dia_sem', 
                         'flg_fim_sem']]



Criando DIM_TMP


In [44]:
print("\nTransformando Fato")
df_fat = df_silver.copy()

df_fat['srk_prd'] = df_fat['product_id'].apply(gerar_srk)
df_fat['srk_vdr'] = df_fat['seller_id'].apply(gerar_srk)
df_fat['srk_sts'] = df_fat['order_status'].apply(gerar_srk)
df_fat['srk_tmp'] = df_fat['order_purchase_timestamp'].dt.strftime('%Y%m%d').astype(int)

df_fat['srk_fat_vnd'] = df_fat['sk_order_item'].apply(gerar_srk)

cols_map_fato = {
    'srk_fat_vnd': 'srk_fat_vnd',
    'srk_prd': 'srk_prd', 
    'srk_vdr': 'srk_vdr',
    'srk_sts': 'srk_sts', 
    'srk_tmp': 'srk_tmp',
    'order_id': 'cod_ped',
    'price': 'val_uni', 
    'freight_value': 'val_frt',
    'total_item_value': 'val_tot', 
    'days_to_deliver': 'qtd_dia_ent'
}

df_fat_final = df_fat.rename(columns=cols_map_fato)[list(cols_map_fato.values())]


Transformando Fato


In [45]:
print("\n4. Persistindo no Data Warehouse (Schema DW)...")

cur = conn.cursor()

try:
    # 1. Cria tabelas no schema DW
    print("   🏗️ Executando DDL...")
    executar_script_sql(DDL_FILE, cur)
    conn.commit()

    # Função de carga
    def carregar_tabela(df, tabela_alvo, cursor):
        dados = [tuple(None if pd.isna(x) else x for x in row) for row in df.to_numpy()]
        colunas = ','.join(list(df.columns))
        sql = f"INSERT INTO {tabela_alvo} ({colunas}) VALUES %s"
        execute_values(cursor, sql, dados)
        print(f"      ✅ {tabela_alvo}: {len(df):,} linhas inseridas.")

    # 2. Carga Dimensões
    print("\n   📥 Inserindo Dimensões...")
    carregar_tabela(df_dim_prd, 'DW.dim_prd', cur)
    carregar_tabela(df_dim_vdr, 'DW.dim_vdr', cur)
    carregar_tabela(df_dim_sts, 'DW.dim_sts', cur)
    carregar_tabela(df_dim_tmp, 'DW.dim_tmp', cur)

    # 3. Carga Fato
    print("\n   📥 Inserindo Fato...")
    carregar_tabela(df_fat_final, 'DW.fat_vnd_itm', cur)
    
    conn.commit()
    print("\n   🚀 Sucesso! Data Warehouse atualizado.")

except Exception as e:
    conn.rollback()
    print(f"\n   ❌ Erro na carga: {e}")
finally:
    cur.close()
    conn.close()


4. Persistindo no Data Warehouse (Schema DW)...
   🏗️ Executando DDL...
Script DDL executado: ddl.sql

   📥 Inserindo Dimensões...
      ✅ DW.dim_prd: 32,951 linhas inseridas.
      ✅ DW.dim_vdr: 3,095 linhas inseridas.
      ✅ DW.dim_sts: 7 linhas inseridas.
      ✅ DW.dim_tmp: 616 linhas inseridas.

   📥 Inserindo Fato...
      ✅ DW.fat_vnd_itm: 112,650 linhas inseridas.

   🚀 Sucesso! Data Warehouse atualizado.
